# every word in transformers use vectors to be represented 
# these are made by asking questions example for a word horse lets say we ask it is it an animal then 1 can it fly so 0 s and so on so it s given by [1,0,1,0] this was erxample is of dimension 4 but it is usiually more like 12000 for chatgpt for each word and etc
# based on this it is known as static embedding after some wword come into play they transform each other according to surroundings word -> contextual embedding
# transformers process parallely ie multiple words at once so fors  tehy use static embeddings and ten use positional embeddings which is anotehr type of vector to denotwe the position/index of the word pposition embeddings are calculated by some formula which we dont care then the positional embeddings are added to the static ones and get a new vector containg bth the knowledge  
# so each word/ token has a key so when we want to print the next word then the n-1th word will act as query ann its value (wq * the words value)= query vector is dot producted with the other related keys(wk * values) = key vector providing values and then based on that we take the most suitalble ones (multiple)
# then convert those evctor into probablitoioes using softmax function and u do some other things and get attention of it see got for formula and all 
#
#
# for the words itself it breaks them into tokens which can then be used in future if  anew words comes into play as it alreadyknows tehold words example play  + ing instad of playing coz it remebered new words if it had stored the words itslef instead of splitting out of vocabulary i.e oov problem would occurs and spltting of word sis knwos=n as subwords
#
# byte pair encodin - bpe storing commonly apearing words ex ggood morning
#
# BPE merges the most frequently occurring character/subword pairs.
# WordPiece merges pairs that provide the greatest benefit for modeling language, not just the most frequent ones. basically smarter BPE
#
#

In [2]:
import numpy as np

In [ ]:
# simulation of attention mecahnism
seq_len = 4
d_k = 3

Q = np.random.rand(seq_len, d_k)
K = np.random.rand(seq_len, d_k)
V = np.random.rand(seq_len, d_k)
Q,K,V

(array([[0.10153645, 0.50961469, 0.07166137],
        [0.98977288, 0.10967559, 0.83762084],
        [0.9992489 , 0.51600896, 0.85718735],
        [0.93177337, 0.96015311, 0.44734972]]),
 array([[0.86408237, 0.01867826, 0.1639363 ],
        [0.42535601, 0.8521121 , 0.86786629],
        [0.67903518, 0.94132129, 0.32157158],
        [0.9458829 , 0.02278554, 0.04280726]]),
 array([[0.93001074, 0.7899946 , 0.43971311],
        [0.64358946, 0.57871189, 0.45127819],
        [0.56800723, 0.74399816, 0.02338687],
        [0.8508599 , 0.31526805, 0.20158719]]))

In [11]:
scores = Q@K.T #dot prod
print(scores)
print(scores/ np.sqrt(d_k))

[[0.10900247 0.53963047 0.57170224 0.11072106]
 [0.9946103  1.24140463 1.04468564 0.97456451]
 [1.01359562 1.60865801 1.43990248 0.99362382]
 [0.89639979 1.60273322 1.68037443 0.92237592]]
[[0.0629326  0.31155579 0.33007244 0.06392483]
 [0.57423853 0.7167253  0.60314953 0.56266508]
 [0.58519971 0.92875913 0.83132808 0.57366898]
 [0.51753666 0.92533846 0.97016463 0.53253398]]


In [12]:
def softmax(x):
    exp = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp / np.sum(exp, axis=-1, keepdims=True)

In [17]:
weights = softmax(scores)
weights

output = weights@V
output

array([[0.71651748, 0.61964539, 0.26772232],
       [0.73871236, 0.60813166, 0.28748892],
       [0.7137932 , 0.617435  , 0.28007072],
       [0.69702657, 0.62723154, 0.25826634]])

In [26]:
embeddings = np.array([
    [0.1,0.2,0.4],
    [0.6,0.8,0.1],
    [0.3,0.9,0.7],
    [0.5,0.4,0.6]
])

q =embeddings
k =embeddings
v =embeddings

print(k@v.T)
o2 = (k@v.T)/ np.sqrt(3)
print(o2)

o = softmax(o2)
o

[[0.21 0.26 0.49 0.37]
 [0.26 1.01 0.97 0.68]
 [0.49 0.97 1.39 0.93]
 [0.37 0.68 0.93 0.77]]
[[0.12124356 0.15011107 0.28290163 0.2136196 ]
 [0.15011107 0.58312377 0.56002976 0.39259818]
 [0.28290163 0.56002976 0.80251687 0.53693575]
 [0.2136196  0.39259818 0.53693575 0.44455971]]


array([[0.23247576, 0.23928457, 0.27326556, 0.25497411],
       [0.18786378, 0.28966637, 0.28305347, 0.23941638],
       [0.18903835, 0.24940499, 0.31784542, 0.24371123],
       [0.20671916, 0.24723511, 0.2856243 , 0.26042144]])

In [24]:
import torch
import torch.nn as nn
import math

In [33]:
class ScaledDotProductAttention(nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, Q, K, V):

        d_k = Q.size(-1)

        # Step 1: Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1))

        # Step 2: Scale the scores
        scores = scores / math.sqrt(d_k)

        # Step 3: Create causal mask
        seq_len = Q.size(-2)
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)

        # Step 4: Apply the mask
        scores = scores.masked_fill(mask == 1, float('-inf'))

        # Step 5: Softmax
        weights = torch.softmax(scores, dim=-1)

        # Step 6: Compute output
        output = torch.matmul(weights, V)

        return output, weights

In [35]:
seq_len = 4
d_k = 8

Q = torch.rand(seq_len, d_k)
K = torch.rand(seq_len, d_k)
V = torch.rand(seq_len, d_k)

# Create attention layer
attention = ScaledDotProductAttention()

# Forward pass
output, weights = attention(Q, K, V)

print("Output shape:")
print(output.shape)

print("\nAttention weights:")
print(weights)

print("\nRow sums:")
print(weights.sum(dim=-1))

output

Output shape:
torch.Size([4, 8])

Attention weights:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5753, 0.4247, 0.0000, 0.0000],
        [0.3859, 0.2831, 0.3310, 0.0000],
        [0.2972, 0.2109, 0.2616, 0.2303]])

Row sums:
tensor([1.0000, 1.0000, 1.0000, 1.0000])


tensor([[0.5449, 0.6586, 0.6467, 0.9458, 0.3887, 0.9309, 0.1020, 0.4401],
        [0.5627, 0.7223, 0.4960, 0.7888, 0.2924, 0.9447, 0.0995, 0.6603],
        [0.6079, 0.7798, 0.5437, 0.8523, 0.2790, 0.7921, 0.3389, 0.7270],
        [0.6466, 0.7883, 0.6020, 0.6789, 0.3525, 0.7149, 0.3050, 0.7147]])

In [36]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention()

        self.fc = nn.Linear(d_model, d_model)

    def forward(self, x):

        batch_size = x.shape[0]
        seq_len = x.shape[1]

        # Linear projections
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        print("After Linear:", Q.shape)

        # Split into heads
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim)

        print("After view:", Q.shape)

        # Move heads before sequence
        Q = Q.transpose(1,2)
        K = K.transpose(1,2)
        V = V.transpose(1,2)

        print("After transpose:", Q.shape)

        # Attention
        output, weights = self.attention(Q,K,V)

        print("After attention:", output.shape)

        # Put sequence back
        output = output.transpose(1,2)

        print("Transpose back:", output.shape)

        # Merge heads
        output = output.contiguous().view(batch_size,
                                          seq_len,
                                          self.num_heads*self.head_dim)

        print("Merged:", output.shape)

        output = self.fc(output)

        return output

In [37]:
batch = 2
seq_len = 4
d_model = 8
heads = 4

x = torch.rand(batch, seq_len, d_model)

mha = MultiHeadAttention(d_model, heads)

out = mha(x)

print(out.shape)

After Linear: torch.Size([2, 4, 8])
After view: torch.Size([2, 4, 4, 2])
After transpose: torch.Size([2, 4, 4, 2])
After attention: torch.Size([2, 4, 4, 2])
Transpose back: torch.Size([2, 4, 4, 2])
Merged: torch.Size([2, 4, 8])
torch.Size([2, 4, 8])
